# Pika H3 Flex 官方网页工作台
选择 A100 80GB / 高内存运行时。若已有测试正在生成，请先等它结束。
Colab Secrets 添加并授权 `CF_TUNNEL_TOKEN`；可选 `HF_TOKEN`。工作台无需密码。
运行下一格，通过你的独立工作台网页连接后端。默认无需密码；已有环境与模型缓存会复用。
Cloudflare Tunnel 服务地址应为 `http://127.0.0.1:8188`。首次模型下载发生在网页提交任务后。

In [ ]:
# -*- coding: utf-8 -*-
"""A100 官方 Pika 工作台一键启动：复用模型缓存，启动网页后端，无需密码。"""
from __future__ import annotations
import base64
import gzip
import json
import os
from pathlib import Path
import secrets
import shutil
import socket
import subprocess
import sys
import time
import urllib.request

PUBLIC_URL = "https://mcp-mac.id96.vip"
STUDIO_URL = "http://127.0.0.1:3091"  # 电脑上的本地独立网页；部署后可换为项目域名。
APP_VERSION = "1.3.0-c45ceda-dirty"
PORT = 8188  # 复用原来 ComfyUI 的隧道服务端口。
STORAGE_ROOT = ""  # 空白自动检测 scratch；也可填 /mnt/local-scratch。
HTTP_DOWNLOAD = False  # Xet 下载异常时设 True，后续生成进程走普通 HTTP。
BUNDLE_GZ_B64 = "H4sIAAAAAAACCr19a3cTV5boX6lxT5YkkGTJxjycKKsJmMANBF9MMt3X9tUqSyW7YkmlVJUMbrfvMkkAO2Ag07whnZBAoNPNI90ZcGwI/2XaJcuf8hfufpxTL5VkQWdNshKXqs777LPfe5+5Hr1q2Wq5rJl5s1619YqWrs32DCo9v1He1aqaqdpaUSmZRkWxpzRlRjP1kg5voE61qJaNqqa4DbypWJpdrylGtTybVKqGotZto6LaekGZ5KZ0o5oeq4714L/O3WVn8amz+ETZZ5TVCaX58lLzzvlfnt90Hl1vXP3JufDVxuWvlH2HD/3y/PzGub9t/PXcxoXHzjefwk+rAG0VphTn4XXnkwf/XPiEWxyr0jjz+VLdrptaPq/olZph2oparRo29W6NVcW7KdWaKusT7u+PLKPq/jAs0VZNtbGUbGgYfrqFrKm6rZe9n/WJmmkUNMvrw5r1nnFlx6r7ju4fyh8b+lDJKbEdE6XdmjqQ2b17T1bd0d+XzezUdmVKO3cNFPt27yr07Spl+vozO4uxseoRqHZY1ssUi6WBAbW4K7NzV0HNart27s729Q30TxR3TezYmckUS7t3lXYN7Ilhd+8fOPQuVJqLWbZhqpNa3jQMOzaoxGJJJWar1jQ+230zKv42Ncso13Gd8O2O3Zkavi3Weefg3QD8VK2aVqAmsjsH92ABS9OK8HtHHzwXavW8USqVDRVfHTfrGryEdanU7HxJL2uy65JuWnZer8CQ5KuyGn5jaiXN1KqwpvBmdBzeTNl2LV80TlRFBwfUsqXNj1XffQcmmc0o27YpewDAqkWtpAA4x1Vz0koq25JK4UQx9z5Aa1LRqjPiqWxM0lNicKyqwD9YGJop65Ydr6i1uGWbSXqZSHCBmqlX7Xjs33FsSiz9kaFzD4mkUirXrakczleUhTMBu56jfU9XDABAo6oX4uKrXsLeFd1ScACif6rmQlHaGz8OHv7jscN/8GZKK0z7u9NgHXzNnNDtKcWoadU4dIObFsPKBaOoVydzsbpdSu2OJRTVgmNQLZb9NXmeRgGG7hvLMLXVbjSWXTTqds5f/tDwEL3XTNP/fuT4/qMfHE8qtnbSzjF0hIcFb0zTMK0cbH+trBY0eDNRL1n6H7RcNhEcqG3OhoaO/5QMEzaRMBNNJc3jiyjpbSqWx7EUcwh5LbsZ/oeXLX3C1G2N6nYuR+3FI8qYuM40xhOqbkeVAEgxC22Gbqq6pSnHGG0P4arFSzHnixfra3eduz80f7ynaCd1OzdnFuZ/eX7LeXS+ceXHxrV7zstrgGbnADLmY6EetZMFrWYr76iWNkSPcO4HIwdFg64Z5XI8EQHGYWhK25pZ0atATOJtVip6KwNt0BLhVBHYsgNtGhJT8AHdca4ydLKmm1pxq16mdZxTN0OJ2k/ckeDrEky7XI6anA8204WyYblrwyBZim1ef9p49F/NhWuNa09xx8K4REkxmhlMZ0vzVhhuGQsioswXjGpJn4xLRMc/AfKKesGOM5GQXauAcBFvIamL5wln5/OJNJGGGRhhWpQAEJgsGxOAf+HdpAZoURYGxILYiFpIA6KQsyqUsEtRvRfoWDbPpB7ofGPx0ubp5Y0Xj9JIh2MukoQ6ad2iht3RM5qs1crIh+SIcKdxllYcS5sazBfRSzyEWFKWPhlL+PYMmoeVBOAlBgaoTFy2mqSFSSiAR4CjcV/TesNvXr5EaEf5MH6oluviKMZ4Rs7L05t31hrXHzuPf2r+cAfYl/WVy87F75ufvlh/8XLj8gNFUOn1lYcbNz9zHl6DPfefS+4uXa8V8fi4g/EKuOBC/TW+vNdYvYTgAssB536x+d2ZjVtXm5/dbNz+2+adp5ufyh5/eb4Uheu0k0jgaW3nmDUYVOJEaJNMb4HcxXTxnnG4+7ocLE6tIrUvB4rL10Dfo5qf97afpj7KjMo4b1fVHaBvA1oXH6vAki43Lj9unD8VS7S06WN23JbnXKYHmKpabL5jD14DCmwn96NgfQBuqu71ac/WtLjs2OWnxglzctc2Ahs+Dihv5ZTWovg2O9BxPLKw4kKcMvDfC3/KDigAVoj5rzyJWAbBzvmWoC/LbJ1k7/oH+/DPjsF+/DMwuIM+Dmb55QAXoZd9XGTPYHYn/+3LbrGG3HvkPgXWjFjMyPXK+NeLiylvKX3ICe7s79g3Flaa5z91bv2I/T/8duPOI9k/shDT2iwuSDzA07bwoH40EBo1NOANecIwylthjFJsDurMexvorHzqPLnsLDz3rUscW3IXxs9Hj8OZCnz0cdTjiYTybzn36IwGT9Z4x4Xa/O6q8+RnZ+V+4+rZ9bWnPLr1lQsKnTFAa87q5ciF8/P9IZ4/yO+HFtG3fiC7FZU4LiARJP/KeoQhGhUfgG/vG/YBo14tBtZ3UJnztePyQICNkP338IMrfIy7ix8iGFglSQIDEYt4CF/lci6Oo9WnzcE6iY7rvb52WuFqEhIuXlr/+ZbijUhxFq817zzwLzp8xEXH1n2NT+tVoGaWVkuSKAvTgwJIwm0dUUU8NhhrpYhQHGeDdT2s4G7bjF7UDHxQ60XdiM3TxN0Nwl623JkAyDsXP2kufMqg1bj2dePKItIuGKZvY0Ccr4qNkXzNJBzKmqmRIOSyBnq1ZAQFF/qeBw6rVrfjo7EqjF5XU1ZFxxmkUh/XNXM2BW3lYLLayWRVrWjJilYxzNm0bdhqWf6oW8AWFE19RjPz8J+FSJlagLWvqHauYM0kq8YUMB+aCQ/1qm4D3PgEnaAI+e7wBwr3qFCXCnWmHNHfSSrYFT9xf4NjY9WYsp0mF0Gu6VTBpPEzcJOmXgOGzAIewUa5BLiz0cw4/47Hkj48gvgMuAasPdoHeOotZddAJpNpgcyAeBFzXvzJWVpWcPyINK8/3vzms8btr4DCKLszIIU37nwLHE7z8TNAps1Hd1mpA+xrm477oeO3lR2ZPTu36Bc7dJ793Vn+Gjim5tPTzZdnlR3v6u/8c+ET6Mw5veicut14+I2yz6iUZj84BCTYOf10fe1q48KDjUtnnJ+ebp79vHH5Z4RnDY7hNNJEmkXjwR3nz+dgzM6Z5c2zy81nX4CoBJwS80swx+bZ753PHzS+XGi+vLXx4Bx0KOcCoCF5aJIB5SoPAmRkEwExlM5GrBehsheq4V7FEj5eNbhjkktVZ1S9rE6UNdpeOw41R2NHtMpe+T4mdxY3OaFsU7KZvh1BGcK5+BhWbOMfaxtrXzlnTjsPr+Px8lruVd59h+QH+BvFDcKGeYXfUrKoZMEqW4AJ9cr94Ro+/QfWfPcdhg3YLt4OsUdiC241X95AonfpMSy4c+PB+ssvnUeL6y9ubJ69iBsEm/jln1lkaNxeWH9xofndqcYPtB2MEyytDNQtL1RdIT7dx/MEdGF+2leYAimsKqWf6OKJNBBRoEpwTM24TzBqQaPcGOLCom7GozFhcNX8HeHxQv3m7Qdize781Tlzw3n0580bpxvnP2m+eNF89tj5+bNYtAIIZAU1KBy1R4ogoRYrVZvxGclf9MTf8fn43mPvDh1PHhg5/vvhoSBW88spsCg6yihIRUfHcVPc0cDmzADdtOO6rVWs8FrgScEPeFKoQISwDHIuSJlya7AUkll8B1sSoQPCQWAplk1LFjJmOJdYtI6lZHmkzq7UShZpPrUZ9xnXjpSdsxa/sD6uq9YUPM8TjxIrAUDEZCMlC1b9hNa67d2pO+oWgAESMlIxp4u6NZ2nd3Gecgf1BMyFSjL5Ut7OKf0ZPrI0TMNKqwUEAtFSEt/8R/7oe4kOwwnublqt1bRqMR7nfkqmhvSLx9VZJ3N0hAC9kwZGRfV5+K0HOrybhSm9XAROCLZhdNzfJxdE2EeoLmuwWQhOAQiRp9OdT/hg1qtwJoolOgRTx1ugK+rgNm5/3/h2ofFf55zFJ4D64KAieXk7B0uPNPH235zbTzZuXWditb52QSmWlObPf3LOrjKZVKRFg880YDo+7oofIwCCBJyw/vymoEiXfwIUu776BTcc0BhINFZRT8a9iSaRM8+V1cpEUYWV1s1B+j+SjtGs4HHbQx436lEFYDXkuzRpoSxUOgORA7Rpa1W7l7iXWGILSgELsrmw1Dj3F14lJWjbkbM6YZjTyJvzxHqV2LA+reYP9ucPlLWT+aOlkl7Q1bLQGxVUQHCovsh4QhM1APWmSrFEFEpGFFQyysi9KXkggwg8eJDhdJxQy9OB+hHYC5k4OvdYL0rVx8w34S7uJgGtYa1IbIQAilVwpNZsBdiC6dfEJGIttue4PdgoYjaASOp/0H61w2pqH9dRtypArm+AEE5S6RsQqCclBhJScPImk66KdnZe+aOy8ZfVzWt/V+Y87BLgUaDEwQPKxvM/QV2Q43h6XTAxvubecsfbGTZLMRrLj8zAIAdCzC9wvBuPrjHvsbH6nTLnzj4wTp6HH4DTlWkEO+0kyIt5I2C+ESIOlvKYmQK8jSOMyK2fQbkJ1hiAUgNJxgTaPimKEGWT7H9AXKVKvom2wgtZOicNY7KspQtkixU2S2RxEJkGi8tRxOVndxAkBLeMwwdT7WwKfkiihaA+5EpoVQvNuTMw5ziAiZ2vzSaVGbI80f/Khsvk1WZhYPgJz+qEDnIT/K3N2lN+pXJtNk17YAXOVE2v5YkxCgqQRA4Q19VmUWWZqhAzoNeYR5Ki4LjfNqfWyPzM/JOwcoXlQDkU2WmaJ14wihqqDTJhosTLUpsVzB6tCFRuM1axSt6A3Qr/4rD1kte3f8j/FhwyFAMYBbDQ6noxTrqPjFTYCcpyYkovALWATlPIw3XDHm88X3WefKEM03Yq/elsH281qw4Fm/xoqfntaYW3HIukqIhz6QILd7EWvXmseefe5otL4SrodfDZU6jnLN5y1lYBjtZfPnLuLjdv3Gyc+4pdEdpZCpmFkFODFWfdPa39xx8HVh1t0Ai/HSoL/wqqPSvbSEqwdgfcoVlq0j05AiSoTpLIOD4mtqjuq9kRjEJVPbjlszxRB9YtXzAqFWBL4oWynoQyNSMpZDNgRwn83PNMGkSfVs6vUhwnAw2MH1vAg66dVCs1IMB86qkkPceFFi5tn7RjEXYFn56OTVdk9qDSkrssVJC40eGCQdPBSlHtZKgx+lKBM1FO4bBwoZBXSR1WJ6xeejrYn0KuhZfQ1JBnJZHLdfKgDzz8lNCd8i9Rw7VbJCONGVTKtRzwDkfYHqiY0P4nW8wB9JXU6cEGhCI+KCBiAbFv4y3Ctl97HpC1K0UpSUBbUColSwVVyXjG1ElSKMfDKuRUil6khBY5CTvt0ypTAfwtv7dXM4ewD44NMB+ObRR7T/rLjrdqXaN0xqGpyuZoB1G4gP+PhxSclaI8JjWAHRWQsSDzFeCn4z5FBsg16VpZtVH5iMg1Bhxi/WRsa1Y77HR1GOuRHsxZvuKc/rTx9XNW2cX8bizE2EU7snQwJQc1tAE+PlpR4y2rZKdhO41JkrYZvOgRRHKBhVLE/pFNJsiWS3admJIObBeShgA7VTBqs3HvozS6HjyQP3j0yFAOAd0vCSQV/PLBO/l9e/cdbPlMf+oTotjvho63L3YSSaDb2v5DI3vfOTyEVXKxbABjBY1P4wJdZWANhg8Nc/tQ/VigD2+loIvjR4bD33FB4cvw748fPPr+B++/88GBA0PHhvZj10ll3wf79+Y/PDRyCAe0f+jDQ/uGRnLUIX0BpPUBfDh8dO/+Q++/m4sd3vt/fh+jto4e23cwT2X2Hj58dF8eYSMXY90ZKhPzljaJcG0N4oZIeLONaZJcBfcbgxU5fvS9off9hkEs4tts2KdRr9w4giqWcOG3UpPQC5Mu4UM89sbv36i8UUy9cfCNI2+MxBLKdkAVqGPHZaGi+L98FThE5Q0lm6F/xAjQbyunyLUj+ISHUgyoZGqOuptPw9tYSNJ5dm/9xe2NW4+cF1dQ3sH682Nj1ZA7TgRDUdEtS69in6N0LILnY1InHqGGMjPZ2EqVmjaJT5MF0ldV1GlYXNe042e+6HR4CFt0tAUb1zWfJlTOz+45p5/B/HB1cTzsNSf6SvyPM03t14s4kxSw0QCTqEvwOKxUUevEXQnGCoEW0LJ2UivUbdKWC0QtuKQ8Kv1HB/uI3Yj3J5Vsn3CUCXLEHlsXCwpyop+tNO+kH2fp1M8qu+Ir6qqWfgYSgAp3V/nCFkY/a40aLL9nrs+IQ/yWdwYso24WtFhgsFgkSsqi94yQW7ZP7I1e1SVTgaUT3ey8qJra569IHiYVg2FILZLx3jD1Sb0qzfjWYG8vVAUkDZi/0hvNqKWx8Q7br6nFCDGsw5BmUkDbLRrVwaG9+7HtLYQv6SUWlu+pbziV0q3YbwDQTXu2oy203QBRR1S3mHuqGbCzZVWvRtkyxSCoJzm0LtBDY/XSxtenGBU2bi+x3q9x+Sfnc2HqWH/55caVGyB6NZbOoSXo7mcbl8542k9yZ4p1DQIlTRx1ZBbLtmbmJsrGxGDVqAoOsajVbCqQDQCIXNN/BfxoxY26LTuyVRrK1k3PMGciTxj+TNVm+wEn+PQc3ehFXFZmNDa89/hBIo9S5JM6EiR/gOdRQYim/+3E+ZCinar4jCYlwNmaSTQN2hEu9GlrSu0b2BmPu87tTEdrs9hwW/jz6VQ+DNmU0uS6p8WxvivcCS0bsQwsmrH9dGLWJptpeko7WdQnNct1zKyo5rRm+pbRx02nsO6s39uQXkBh8kHziCK1EemD2KpIk034rG6ivs/SmyCDm1xGlj59L1p0ZnHPWyKpvKfN0lOirQIN/YVxGAGr5itptaj462m0iKx27Mk2jLLVC3JfeTZf1EtoNDOtPNXTAASZVKfcIaCvOesHXn0wcjci9WtkDKNuw19CZCy4lG1UIX5uI1XULST/KWS7Be0XE6LP9dqkqRY1X22KmqGFwV8npjSt3A278y8Nw4yx4iXyZEUvfKdhbLmtr95k+5lVDcTZTKE0MY/XWq9WSA9WBRieQE33WI8M4gFyOPWmVBwex1+DsSS9TeelX1A+n4yh3ON+Ee/ThXpRTbypwFHVZFv0DrGL62MRT8j2fQUAGeeBCdULIJMA0x7PJKDUyRyXMNBnJN7fl+zvSyS5VC6G1WLJIlq9RbEJ8rvJ7nTbj5/87UngLOqVeCKNttU4Nurr1JqtFqYAXep/QO+GnhZdYIH1UhNa+wUslPUofTwyV1P9qOKK+VolZSBs75RWrnUFLwK3UhAGI1dCvMV6pWbF5wJYdtCPYnHn4bTnXd3boEuR5xMRcTIBJQ1JzYLAoiZwrNqTVHpgKpMmoDuMoavCmDiEjsPSjk7Afs9oFD9nCEslKv41cg5CoymgMkUr6jaKeSc0fXLKtoCPQMtWHc2Q5B1hKcxmp2WsmwDJUr1aoNMXimTbOnytJThN/gCYsVARNXxs6MCh3xH8jw4fem9vfvjY0XePDY2MjCs4gN8oIlLv9gPyTDmvHBnegZp2YNIan3/euPLj+sr37NfXvP8tsHzrKwubp5YbV56gK+mVJyiRXPzeOXWz8fWdze/P/3PhFDJ4q/eds6vra2vrL66sr9xu3nuJ4X3Y123hTrR59nzz2T+aL882X96AFilmcBFrvnyEJvFvgbO8vr7yObrSs6sQsZu/PF/C/h79ufFf5zZunwLh6//1Z5SN+/9JzX84dOzQgd/nj+w99t7QMT7wILvByciSuJ0nr3MiEcV0USPehNwgcxl2qy2ZcCqtQVwV12hV0e14DVgkoE7wDa2PSWXbNjJsWZ56Gw+iWOftig98SZ1G1XPBRnKtjSUlJ6haBV3PSWf8UDCIsCraqHAjoAHIhwMcaFyO67cuVMHpUmE4soI4C9gSfqhpxfg2EWO3bfoExcL5NTOtaxCgz3SgZNMtDQUto9ybbyKo6ocDkzf4eJm+Sfkw1+vMSASrIr/VYXLsX5NTylo1zl9Hx3osfbKiWvmZsZ5xjBbJ+qM46sQ0Z3xBLFvoVt0lhHbFbMcA2Yz1OM8XnPvnGLw3z55tfP0M34MEWytr0GAuI/xJc/T/RNgTyy2Yt2ytFq8YiGaSAOOoYQ1bYOQ/IC+VjQLMmCYS/PYb5ZhGqEO0jEEIaglELdIsE/cN6Fe3ptAHBdmqimZq5VllH3wFNoXZsGCTQeITVM7TSm7P+VeXPeBgXyOXU8ZJIfwEarQubmmsh1cUfcShn/neOVrE+eAK07fAKicj/CRcGLUAZ1SLVs5EX/S4GGhSAXrdqRaIi9E1gYjyImxT4tR5in4m0O00sNscfggr4oI0cLuTOmy7mQfEBlBbzE8ZxnQ8CBKJzkJO8Nh2OC0+95rIIDgRHYlKGs+Dst2Bp0A2STbzgmwKKSBPcemzrngmIsx9JKNx62Vj+RuP7EqqIGiJc+vPzcdrTIg2P/mW6QqRBm4QCdWLH5w/LaNnMPkBY0zNgzvOl9fcsK5W0rO+xmRv2bl4HQpM6rbCyhWFVRnwuvnZReoGO2ks/YxlF580vj7bfPSkce1p8+VlGBmLpNjD6fvOykrz2RdeiPzSoltdMgYkvjAp5z8oo3OoOoFXTcOg0+CnNPqA5vETnAfkz/JT/Wm5WGM9Pq8vrC3iPV2LPrxLs95ksAW9+8RpZmCkG5SvViIqam+sh9gvf+dBMi1cLbnVjh2zHEJaOuahRGRxPNBgksk+1RtEpOYHCvQ9v+06ND9kqHHHxigUMQ/yTukj9PM4hgJ1WE2ulJZhkzg6b0lCRWpqYRr9APPEnIgm5WajyplOMGooxFyT/sZwZlgIuk56bSKLkc9LO9+sleYvSMZax4x6I/4eOKXIXXhHtKLqVfcMSuSKiNejXItfIuU6f1XwZuSdD6fJWb3obbRP1gq8EaPCrakaH6uDyoEdmSyczdOLzudfN1+8cM6fhkPSWFpufvrCeXRz/cWy8/ASH1IxyWkdcQoS7W6QSad5BEFDggOdENEJqfaDJddfftk4f0rQ7bXrzUd3oTUfSsSqwUHkRGuJqIVwtwcD+d2tqrouXbJYwTA1WRWpCYX+wFSStAzo8aghJ6ky8zSjylMjWpQu7nkUXG2LFIjIRLb5jsvDT7xU6Bj71T30n1k5h5vEG08u/xtrn4Fw4G68f2zpg/375M8h4rxNoRV0+++uOA7BX5IHtfH8KmDmxsVLG3dXm4+X11cucBjSxo9fN7685I6I1yfNTiu2qVYttLiTTpHH0K4A9oHfPIAhCF1fWeOZQ4ebC59snl52bjxwe/N2IU08gJb33uTLarHIHbcwvl3US7YR7KW6WdVgBfeTZGNagBvQBsMoiabZ5jtOjMSh0DRZ3MMIjpUL/Lz51T82Lq+5M8X2WGyv1HYEenHfEs6qBtq+tNhYvMRC5k9PGaxC7QbBvqxLqEfEhIdEit95fMFVAq9CZ75oIOrDvpv3z2x+8wUGpDy8jnkUFhlG8F848nnSyuB6IXbOU1P5/FjPoNRTcMuoKOAtC+gH0NbkKQb2HT6kfFzX6hppKGE/tRQjcUC/E4DgYa0w7i4tbXWciMcsKkDKTdWcpew76V8hIc5URS1EZ8epAHstlASvki/H9FQMbN232qbS0SeBXeyUWceeQo4BQSOowqBBoGXPw3fGtK5Zvi9p3gJZAFNdHDx+fPiYBqtu2QeJJYXzclz2gB9HqIpopG6WkW0iG56LkfFH/mNAfvCVfiBo+NP2jPV0m7cH923v8HAemJORQ0ffp7rZdH86kyrsGChoRTVFFjcsdnzvyHsjGKc/1oMeawBxLZH6wNPIL8FYfUBQoSoclu+jR2M9pXKwsojcH+thd7nWDudh7CPDQ/uO47CAk8Agcj5CGBXOT9nBLD9gpDk/9Q/28cOOwX5+GBjcId8MyDLiTZ8sgwHmwJqMVYd+d3zofVwssRjk74WDg+d0TSKQ9Ec170mTjye0idpYz3yLJDbWQyoe2QqgJFEBhBWvagWrIsXDQynLnlBliUpNDDVdAoZTvtwBCzcPK3VoPwXkpiXvZo71/N/RTGqPmiqNz/X3zf87IZkje3+X/2AYPWygdB8F82CY37ZtfYyACmUgt8oIoYVBT5shkXTc0sqlpIIRJKSxRKWtNqNrJ4SiyJ9VA0qmuSA9Uml6ElUk745l0GOIouTIHVIUiAqoIEeVsZ6PgF7xAghfLv5Rr5GhDGYaEg7j7mjQvYZjJoTnAOcQsWQCoQjHLnc2aBg7YZhFdvNCV5bqDAyG1Jkjxz/Yf+hofnjvyMh/HD22n8fj0hK3CQsQD1JSDGuXjklWmnyMhPGxvy9cp2yQxc1FU+ljhw3PC84tdkKd1gLFhmbI0y88AtsQfOtWJXGR8QjMh96rIKvPYFcot4WXiLM9RXwpqXoZ6Ea4xd8ojWtfb3z+tLFwqrF0HwRloPS/PL+5vrbmfH4HU2H8uOYXvUFA3Ty7jFGdly5sXv/KWbzWWDm98ZdV1Mc++zsnf0FnEyTs62vLUFWooKxeV7Z1h8QLEFgJxtQini3H62qgZSCpFFWtYlQjAYPrc+RS3FWasGZxRhOHBpbTD5ZMVd3JO4/+vHHhCcy8+fIGzBPDRC692LiFoj67MDSffIqxt5/81Lj+eH1ldX31jMIjQ+caVzmAgbKLT5y7nzUu/+TLZecbEAMgjSmgXsWERrPCoE8WI+m5Fk+g6XwXYIrdO3e4nms+uU1U3Y7oCMSW7UTt01XtRDwM8klR1rXJJ0N2/5Dh3Rs28KB6Me8ffJI98zob0bm7JDIB0pVPBDnTWFHH1VJB5qv51WYRofDCBeZGMGTdt9TEqFHPiMfRa1e0Y6E/uxxdorNhf69tm/pE3dYizftBlYa3xIw80c46JRaYTAJer4H8dtE5LEQNgKGEVOgc2p8u1cvlCqoS+PuWeRxApCLRCVVXj8/IPA7ozQNnv3H1yfrK5+vPvw4gV/I3kfn2AtjeTxXSmGWKB8Hgug1eRrQyWiOSU6Okb/QOw2vSVh0Y65Noo0FOAM468Qu+1UD1PZUnQ06223mGorM7TlLh4DtrNBPs2gVc4YswJUYrQ4elhsvjbkZllfFux7nxwxqLm4ydcdjnfwK5MzBIGaVB3HroAFPOKQItDPUKBEzJ2Aw3Bmysh99FUNJWuJMhDX7Ay2IyH9oR+iidyhL4encwOUWbWcNeYGDt6XtK9r8X/oR1FOfhtY2/fYf6WinuB0ZGSWwCk8A3PAVmqjtOAwt7k6DGxLYRc97FPhEBRLXA84tuHqTQ9ljBAXrRHoJF3p2pBWq4qaAC1eRbrOTPnieSMAXK8jvBrxMP33EVTLTsyEXA8cpgejk4bAeTYgHjS+uEulE5nrZpsNxpUPIr+cnXLY/S61nMRHQuBJGugEbGIC8B3kL9/eKZ5s8PNi6c/SfwONeebl55CUdo4/Ka89PpEINIdMe3cPgmvFhUCrlgzjuKRyOppLJhpCoakxymCWhhQgNcEO/bti3AYso8U1ihbU4sag3+QuWBfrRFdgGK91FjSsmdVqGR5tl/KHsymV3ZPXv6Bnbs2pHZsyfr5RL75fktYHEwyvXmxcbt1cCcf0PZK07dJAufs7zSfPEC2CS/As1ZWdn8y1rj+s+UkOT85sJXzup368+vO6fRoIGQosCis9qlee4eF+RAXOfJD5s3ltEugqAVYBFFBFHQi89NkpNUMBgouF2+gCIXayX9Bbx4omisxum4qBmZfgursF8AoYBRRArjXUFh4/xSY2kBYVGgA6QnbvqtQNdI7JgqImD55heGKr3UEgIcYIT9PITgBVwBOoCESi1YSAY74cqMjm+BIIKZshjr5nyahFCGLCyFlIB/vK1k+7ZeQefuTZFNipZv8/YCUYFsH4Dj90wPUehwk84Qwm1+d2rzxmfOyneRGmDX4GyFNAuZpF9DkAnoADI+YalQ1tQqnevR8eDuRWfsart8vqSUrR/FlmDKLgJgDxWHPkncyHMabJPBNLSsvoVhfoL5u8AiyWxjnGJsVHY4HkZxIYjzBqjzyLFalMHfGsUv4xF2f7HCMphwTnY9KLKfcdODYlj4PD6fCAUCUgfu7o4DvO3B5XM/iI3GD/2BD2LP3Q8Er8TQUZKRloa3t7bZBW/duL0AsA1jcm79DPxxv8KIkR43v/oHPBJc/6AwqGNkx1nAk1+sryw4z7+GShtLZxuLV+EnfOW6UQwgOx0R65ULcmBJ4mxyzOx47EeO6L4k0zn5kIzOGcHUOecSbSBQOfxfUvHFiObE36TiQ805xm/RzXoYOoePSV+evJyAjaB8X59A9X40U0vU2qfwkiZuoQ6LdARvE+gAB4cCboCYigwwV85vfvnN5jefNe8vcvAN6jo4xkYkvfLlw23cXgaGPSRMmGrF4vjJUtplz2kOAeK7dsF59ndgZjav3Hbu/ojBPat3gQA3/o75Zp1L5xkQFGsKs8qcF/1dfQKbPK1h+rOrT5y1VVF5adn7rheDGhmon9fDDBC/DAxceMAGC4qXW3CWorkOAqoo0cJNuYMLquxAzo9nd0ZwU2I8YYZKjv2tFg7Km1U2lHnd1QG26DaAAHIwn6uuo9WQc2ihA24NRCnyxyiZo+26hRiFuWwyGRWZRUEfVNdqVsBVLLN703x3+J4cK78X2//wWwBShgA3eZtz8Wrj6aJzZpkhq4UOCOfJj157nEQfPyLvDneZ2NkRhcG3QRjsbiasbsSzsHZHpCV8uOT8fBpwIc+BpeKWCUCHMqRZBxwlQVBseE78Bbxlang/Rs6nDUq2zZfDS5HzrwHZPeENjwq95fAtH/Mc/0lEEFBcj1ExKnTMgN8RiuU05qpuRfJU2MOGwt3Ap0KKVno+vog81NIiJUVccD17AatIZe1bYkRvp2beEiv0NppNfDrN0FCCOh/XLID6/rGeORjHaEwvxsbnUzNzqH2DFxyWJPNOED7IJubZOOOfVdmY7DClyO7ZPvHqfUNXwb5lcovIrmu8XehjQ9sb4JBcjcqvMzL7pO1fdEHRfe7ytVFPYzPuc3wPu16RaEW7I8fmhxqcYSDXgUyU4RJNN+c7dEdWa8mjxkNZ5hF5coTDWIRLPSr+EFVw5g06KaNSWzPOH7zEG4xVoiM6g143Esy9ZBxY15eNAz/50nHwwTUFoyRGFFTJ4Lj8b8aj+/MrZKjFUb+SZlw07ali8Lv8JSbM6gavOv+WVXmb5Pdgio42KTYiROJAlg3augi52J9pAzeqVaSpRWXaiM624UKPX1Tg+n7pNDGeiBSpePk96bRTeg/evBJOIiC3kLJ5kGwjkaPxSROhmsFhBXN7uDiCqJ2HIkL8S0cmgomTRwi4+njLejOa8QgwSduS9kRsAyISecmAJFWCKgepFfpCEvUkh2l27A7QwLCBptxuNC4H8GrDEUyDHA/zKJxRlp1z11cusMaohbL7r7eRxs+295uwYr7ri07YruhedIIV2+TDRBAoYWg1pkCpxTvl3uyc3q/N3Sh9HRKE/gv3o7SZZJLn0um+nHbWWm8RgsZazy7b0jeZZ4M0J5q7+S2CjV6oaEBxit7Rix57iy9E67pzNgxKvUHAWzMs/WQk6EIxvEWmNhln4CHxhXya0iOH3n3v0OHDlHIWOxYZINyPx4eOHYk6PlS43R0yUVfWhPIBb30Zj4CLYQaKw4YxXa9FZYDkUGMfKqOba8K2ab/fQpgRbcuhSuFbGPmFHObBTqFuIiOR5zfxVv25Z9enLCPyPPQH5H82wbeM+MQUete5wr4cO2bg1OzovmgOdPKyiXafUQNhhjenI36nRQbaJOREA+NW4vGPOslEhO7bI/tEIPPrR4NUVsguSLDa5yDlYbRBCZjoVa/WtUjcjUZPrh0we0Z7pTBx0Fv1he0IgU90FJE9ARnMI1Vnzzh3zwq1Cnkbu47IjS8/3Ty7HEkgOuEZTv8j9m4yigNuj7WZN2a2WAgJrVW3zEsVzj0rr9vo5NW070D++Afvvw8cLWVJavW28vUMxK4WJ66QnAciRwe7FU4XxTECYz1Zv8zRNcB34ln8VL8DdWL5/1AVsJtZrwFISC2AcEzyuJZEB/rd7u472LvXuOiuMy2NJvgRV+O5Apl7OR4yuBTDQu9EfPRr9EanJ1/VTkhHnVw8isgl2t/BF3HdIEgpPKwWebLtlYNRd/e5+9H5Ar9XuY+vmzv5/M4hdKuDl9w6KhB6q8TpWzNwgi6TtBvMHoL9j6IVLLrj8cTWDXd18ro9iZ349Xb/CG2HmB9jCffXK7ZCaD3YBLwVocWRxKMjG9sGxb8arYvgrbtK2dJ9+nuP9JygCBCCSuEj1L40rVWLX2fETgNsYYDMOIYvVbmTrWQA0fTWcTQdxJAydS2CZGTag9cZA8ecKP/7hFbtT314GG1jzftLzccPmw8WeHzOjQeS2nc9IMoNUjPQCGBNqWbxFYaG3g8izgcvYduFJjln8czG0tlfnp//cO+Q4sbH4Od+ttgtCpNen7JdmPSULCctIP/QM43Hy875VXgIWGCiQVWYq02NwjpVEA7MsZ742Fhxe6KX/yB63gJN+mPcUc8o7Jap7PhodjzhmU1Zgsl0bojSNlJbOaW/i4MQ3l0Rf7NygZfGXcBOG+puqrgJoyV3YaeOSy5cCcWht6dyKxdFZDjv39LWQ7G015i5v88uwPeNP/rgFBXBwFbbvVbLS6tXt1/rsDErTRGbnBdwfeUhQu/j75r3T8lslktdjJQCx15nCMGIso6ojSpt0fArE8hfizi2kDR+jL7pN6Cwanv9bICLjbxa9n+aKX8dpWK7S2TKdBeyl74r0hLR1kIfvFCmqBlb2zK6mYyMEXRNeWQqlTGC3bHkNBwaNeXlpZ/AcfNrCjHJ0XPo+o7OI21huubgBeZiQV8Yf2yjyz7h6+Dw59vtREdUFrVMGEri3/BSEJVsLiw4Z1cRp9Jd1UtsrG18dW/j9jlGKq7swxEeNrQX96kNEqOpneggPB59obV78QQKIPBu8DVPgwu6rtKiw7GIbmKLE0GE1e1FBJGHVy9wXCJqIBTBLMOK+PZXUXc1eZ6wVPgzUPm0Lh3Q0RaMtk8ZlGyJSUq24WBbtIg+FS8CRzvbSsh8QrZTUSqgiYR3LZpP8uXvpPuJOBl8lZCnyIuy7YooB2HfhaHMp2a2sWU5vIdiENQqMmOkSaYueOejLcdihhxBwe22ON3IC4ei/ZtYvS5dFYRKhZyW/nvhOz+sEyhRW6QQALo4IQT/UnhlYOe06TjeGJRJRt1PpKSUvh2YeTta119KsxpYpuTqqCVJ+IMlRZRvPDr4V86dA4XznpsP25M50rI3m854dv/fAsDWNIzKdYFQrdVadMx+xwMRCQ7Fwo4LAiELCC5VbJFZx9/Sb5T9Bu0a6iNFkLO8JsJCP+SizpnjZACklW4Je6E20bWWqSngDMpiBdx9UOcx0KrksOo1uv3QP1zfQP0zQnWGmIox8VFSusH0ZTJJMfCWICUYvZtGldOhUc2ILGduPFdLzGa1mAfSVzOqlsSzkWX4xlaY5D6+tSyFWVuYLmI2S71AxvheHMqbIBNiZLeda/WO6NDgYa06aU9JGzyuO04vkdiiNubrT2EbplHm8VSNFF6YoG3V8e9S/rmkjtY4zl+0YVX1Uil8/HkfokwcwaZHNBsax7KcDgufwoPxKlitRh/ELUJLR6vgBxS1bk9h7vUCmiVajo57NzGSilrNDeyNxlOcl8ZF+apJV8jQKUmP6Jgjg6cR75jqCuqRIo5xtpiU8F70loFiAqIxlDvaYCwktCuT7IhXQEzZmJNgjUzwG8pIOAfC8KHIBsHgyMnxsKLsdu1iCHEfWs13iH1Z7o+cexi0x3oywUVA4QvbeAuwC97Sg89vi5sUuguF+OGTxsNvG1fPYnauu/fJe3l14y/BOJOIW06J/hGYEXEgPrmjgym2IXzrux2YvAF9VflfI0ffVzgDWKRDNV2e5q110ci/O3Q82lgqE+EFWRmoMXx05NWqyLfhOq0ALriFeqQlOKDFpCXG2XISGNZoRjPTzFoh1geJgwgxihaSJZ1PKsGIZNGbVJN21Zi06bhRmJSFYUdmR2u7fpfs1xzvQKY/YnXeMdGdeFivySWCU1HVKCfeMQ2oxBZ5uqOEky7nzp7hzpnTm58+cJ5/4qys+MUl4VDOhaTsRLPIBMCEtz0MI3UT9X0yiUlc5FCwp0IhrUmFbk7noml+I3OgxPEVffZXMiw7ZPtkgQmh288+ousNlnUF/CDqGdEKqQOY0T81AkQEGUsWu0zDslIWvRpsj4qjAGlluXHhnnPxWvPZg42/3pAezQRO/aHL+oj1xu561ZreC7yPjsiZnIRgyC0JM090ko/47j7hIYlODWqxSPaMsLXcE82QkLQRz1Tb1pDzQ9dLm4xotmtBw2oykwOtol6jcC3GhSdSNiDpnVE2eskmirbJ8TrbTom6xTI7T75sPr7S+NudxpUnhNFvirseVhacxTOb//mVuKHv8RVe+749UY5bXjIP6odIV0KGHouPEWF6HYKQRaVgeEFUML8s6csg0MKLuN8SnTzE/PsxqtdQina3b7syClsy/lpL3Hz0cvPaI+fxGU4rAdRz484jAcmt/ikF6J5EmjeVEfohste1HrnfpQ5wqlCtmBo2DdtwTx1doQL1hJYirPGIGq8xjYNF9mzeZf9LQX4nN+cuk2SYEvNvUrKZXO+bykHo9Gi1PAvjVisaYoHciG0C+X5TOaKeTO2d1HI7Mzt2ZzJzPMn5MLMbOsYw2TIyMaEtQ25UwlmQMw0f5GrJYO0aSnR6QSjSZJLzUlk7meKcXQyZbtjLoOLL8kR5caqca5HXhyMKfR3jB3zRRqfIpfPyMlosTdGULVCamG/xa8OaESCLU5O6Kpk0yGuOX6BYNolJFOmqBO9zIFjKDYhrqd1ZP8o6Ia9RqSNiJ2grJyK8E1vAHc6jFQb8Qm5cAANvUW9FK+pqb7QbszeNIFRERaRscWApmIbTFfgPb8SZjaQ+Brp2tyM/3R6+0Nl7pXOW2eJoCfWT3PyIUXZF3ERSI6FAc0sHPOA6e7ihhzaeO+GjOjoIgsh4d/vFcxikQcx3O9tu9sNDcl48oqBoMJu+wPZXxL0qaNF1495MX7+9cX/esERvnBXCf6TLEnpYuRYcPLXR6llOcU5qQSSPoELpSWAVa1Y8EeVhjkpFmYjXvzFt47Mkn9+qc5W4SHTuadPbL2rnhRVO9qKjzj2RTrQTpLaDD6o46E1eGiawy87w4ua46QZkuHA8zFa3BQzGYEGo6AAInWbuU5i7xw45A6qI2sIouKAGIy0kbHjaYnWJbLhdhi10buh9r5uVT16JycqS8DKJrGUkErm5yLzc4H+/07i9xBIl5u56cUbcWvESRUs3rTfmFj6/xCFwzsVnaH65vdC4+mTj5mctPhmRSn+cSmu0m9D9y9UkFT1q/2lqYagta5NqYda/HZHhc8HWRFMhD1VywO9gSYiLvkgKwCcfOacCEd6hgjxy210Gm5Jnh/PkZuP2977Mqh0Bg9t/ZShoK0+j/PdtlC4hnGlL6LG5Yd/82F+S3wuuGV9RXgzki8cTAbFOO2nLTIp8Q20w75NXUCRUqGKwXnyavX0xuZetVZFaW8HsUHSLj/AGxy7w0hK3aKLFnVdsFvaxteLLzUzUfHyPT0fj6+fO84tBZfSvoDUU+WuE9jCneNkvu8j/sXzFTfSxeXvBeXJxffUCpsw8or+DB3vlMub7+PRFhCaxZGqUWZqv4izq1nS+7hph3IOWSGO5wHipohjudmXATc7Z301WB8oKwPjlxY3Nsxc5wY0bZy0yZv/0FBMmXjrTuHaflz4wchGU1k2ku3vmW7GHR4wAe3hNbkcICmYgwPrcEIdrMuzG3fRq4QNXcNViaKgVIRnxbF+mo+pdWBFNW1oRT0grYrQjsKlhlmURMgF7EWHipvgOt1wbCX1CZO4M65ErejUusq4mvUYSibZOMwjHE1t4FEWeM0z5s77ysHH1oZtajrQibbpCDx9hUMHeEu1UB3J5UpxgLaowLbe8GoG3eMsrPKhOHQXXaXmpb0Qq1ijGiZOyuKCG2CCct4Vx6CChV/xNG4vMFvydlzyyh6NdpiEpdJQFxjh5ugWNfQkkhWgN+HoFK3hbQkKns5NSWmDICFt36FogFKuL0liK1wYlqW4qi9P2OVuid1DeVKvkGxeBdI/hp7CY5lUKza8dO8k+SOg6ui2R4j+Iv712ovkAbk4o1dTqbDwoT7yGyOwnQ4qYGy5yAM2R5gBGt7UAQ+OKSoxlM54Trgm08kjVJhLdRdnJ+lhHjYygQWyNOEVuqmiddkfGBOKX1sQa1PTb1AQa1vgnI712Kb6C1vDWxdrCgi3WmS5HIkhQtvXO0RlsHz3Vxgoc3OaQTlLCel9m569uzvejAlw3N4d9erKOSQYoAwwJR8CpsYus3/5vFGzNTlk20ILK6xn+oUiKNmt7dgvT/94CmoN41YXZnlZ9y37/R10G5G7Rdm3tOdAeluZoVeZTc1B+Pgqu8IppgbW772e/bmHoksyuUIqpNl6+jHecvkmEAmmKUH6l5ggf4xvoOdalQ0N7P6fIACcRdUQeTzTjFsmuZNOdzi6YhBgiYl+wVMQpl1yLe6FWmGPBeon2FpFOjMoEtDbdzv3O78/Rhv2giYW4jsgLg+TdFOakuEBB8BcWBb3I1+m95mQdt3GYvsgd4XJptVjMq6JAnFI7AIOLAAC9qfWynWtJAv8fR4+9J7W+VbuX7rhLWQUTCQclDckf7M9j0pD8UXFFhyeytOsU54FtEt9BF47K7ndnd+/eojLe5O0f8VhPtm9XOgP/Zt1j0bZjkYW/R+rwcqh+AQSQt9EhRlZHHy3iQqgVttviO/cKNpQKhICK74XsI+75iQcud3UlCO+OchIi4HiTdbTXZwupmYa0EYlLOHJR92zEuVNciCSNNY0LCgKs9M/zN4FiDDTDTnlx3y0HXJEXJOG/brMkrlzhKpSTUiPW2KjSBSGDvb1z7gDmB+fcEdAdgIFbNVtlFzEo+oNX7KHa2b1VxvV0mDBUs+hGqvrTA7n+Aa3sttCVy0lHacvJgOcWcFNYhN5tnc3CV8HLFRBqJRE9aTPvxhu84u048/8fS0pX/4yuAAA="


def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3) as response:
            return json.load(response)
    except Exception:
        return {}


def live_owned(pidfile, expected):
    try:
        pid = int(pidfile.read_text().strip())
        cmdline = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="replace")
        return pid if expected in cmdline else None
    except (OSError, ValueError):
        return None


def start_tunnel(work, get_secret):
    token = get_secret("CF_TUNNEL_TOKEN")
    if not token:
        print("⚠️ 服务已启动，但缺少 CF_TUNNEL_TOKEN。请在 Colab Secrets 添加并授权后重跑本格。", flush=True)
        return False
    pidfile = work / "studio-cloudflared.pid"
    if live_owned(pidfile, "cloudflared"):
        print("⏩ 本工作台隧道进程已在运行，连接状态见 studio-cloudflared.log", flush=True)
        return True
    binary = shutil.which("cloudflared")
    if not binary:
        local = Path("/content/cloudflared")
        if not local.is_file() or local.stat().st_size < 1024**2:
            partial = local.with_suffix(".part")
            machine = os.uname().machine
            arch = "arm64" if machine in {"aarch64", "arm64"} else "amd64"
            url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
            subprocess.run(["curl", "-fL", "--retry", "2", "--connect-timeout", "15", "--max-time", "120",
                            "-o", str(partial), url], check=True)
            partial.replace(local)
        local.chmod(0o755)
        binary = str(local)
    tunnel_env = os.environ.copy()
    tunnel_env["TUNNEL_TOKEN"] = token
    log = work / "logs" / "studio-cloudflared.log"
    with log.open("w", encoding="utf-8") as fp:
        # 使用环境变量传入令牌，避免在命令行与日志打印中暴露。
        proc = subprocess.Popen([binary, "tunnel", "--protocol", "http2", "run"],
                                stdout=fp, stderr=subprocess.STDOUT, env=tunnel_env, start_new_session=True)
    pidfile.write_text(str(proc.pid))
    start = time.monotonic()
    while time.monotonic()-start < 30:
        text = log.read_text(encoding="utf-8", errors="replace")
        if proc.poll() is not None:
            print("⚠️ 隧道退出，工作台仍在本地运行。日志：\n" + text[-8000:], flush=True)
            return False
        if "Registered tunnel connection" in text:
            print("✅ Cloudflare HTTP/2 隧道已注册连接", flush=True)
            return True
        time.sleep(.5)
    print(f"⚠️ 隧道连接尚未确认，请查看 {log}；本地工作台已就绪。", flush=True)
    return False


def main():
    if sys.platform != "linux":
        raise RuntimeError("本文件在 Colab A100 Linux 中运行；本地看界面请运行 preview.py")
    payload = json.loads(gzip.decompress(base64.b64decode(BUNDLE_GZ_B64)))
    namespace = {"__name__": "pika_installer", "__file__": "/content/pika-studio-installer.py"}
    exec(compile(payload.pop("installer_runtime.py"), "pika-studio-installer.py", "exec"), namespace)
    config = dict(namespace["CONFIG"], storage_root=STORAGE_ROOT, http_download=HTTP_DOWNLOAD)
    namespace["CONFIG"] = config
    current = health()
    if current.get("service") == "pika-h3-flex-studio":
        work = namespace["select_storage"](config)
        if current.get("version") == APP_VERSION:
            print("⏩ 已有同版本 Pika 工作台在线，无需密码，保留当前任务。", flush=True)
            start_tunnel(work, namespace["secret"])
            print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
            return
        # Only update our own idle server. Do not interrupt GPU work or queued jobs.
        for path in (work / "jobs").glob("*.json"):
            job = json.loads(path.read_text(encoding="utf-8"))
            if job.get("status") in {"running", "queued", "cancelling"}:
                raise RuntimeError("更新进度条前，请等当前队列完成或在网页取消任务，然后重跑这格；不会中断生成。")
        pid = live_owned(work / "studio-server.pid", str(work / "studio-app" / "server.py"))
        if not pid:
            raise RuntimeError("检测到旧版服务但无法确认归属，请先手动停止旧工作台服务器")
        import signal
        os.kill(pid, signal.SIGTERM)
        for _ in range(40):
            if not health():
                break
            time.sleep(.25)
        else:
            raise RuntimeError("旧版服务尚未停止，请稍后重跑")
        print("✅ 已停止本工作台的空闲旧版服务，正在更新。", flush=True)
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=1):
            raise RuntimeError(f"{PORT} 已被其他程序占用，请先停止旧 ComfyUI，再运行工作台；本脚本不会杀其他进程。")
    except OSError:
        pass
    work, env, cli = namespace["prepare_environment"]()
    app_dir = work / "studio-app"
    for name, content in payload.items():
        target = app_dir / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
    env.pop("PIKA_STUDIO_PASSWORD", None)
    env.pop("CF_TUNNEL_TOKEN", None)
    log = work / "logs" / "studio-server.log"
    with log.open("a", encoding="utf-8") as fp:
        proc = subprocess.Popen([str(cli.parent / "python"), str(app_dir / "server.py"),
                                 "--root", str(work), "--port", str(PORT)],
                                stdout=fp, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    (work / "studio-server.pid").write_text(str(proc.pid))
    for _ in range(120):
        if proc.poll() is not None:
            print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
            raise RuntimeError("工作台服务提前退出")
        if health().get("service") == "pika-h3-flex-studio":
            break
        time.sleep(.5)
    else:
        print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
        raise RuntimeError("工作台启动超时")
    print("✅ 工作台已启动，生成任务在网页提交，不会自动跑示例。", flush=True)
    print("🌐 工作台无需登录，打开网页即可使用。", flush=True)
    try:
        connected = start_tunnel(work, namespace["secret"])
    except Exception as exc:
        connected = False
        print(f"⚠️ 隧道准备失败：{exc}。本地工作台仍在运行。", flush=True)
    print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
    print(f"隧道 {'已连接' if connected else '尚待确认'}；Cloudflare 服务地址应为 http://127.0.0.1:{PORT}", flush=True)
    print(f"模型缓存复用：{work / 'hf'}\n服务日志：{log}\n生成视频：{work / 'outputs'}", flush=True)


if __name__ == "__main__":
    main()
